# 06 · 2050 Projection and Hotspot Identification

Roll the trained Random Forest forward in time — same spatial grid, monthly cadence, 2026 → 2050 — and:

1. Aggregate the projection to a domain-mean and per-district summary.
2. Compute the **percent change in domain-mean NDSI by 2050** (manuscript headline: **+20.8 %**).
3. Identify the top-10 % chronic-salinity hotspots in the projected grid.

**Inputs required:**
- `results/rf_ndsi_model.joblib` (from notebook 05)
- `data/ndsi_per_point_2000_2026.csv` (the per-point historical record — same file used in 05)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import joblib

from src.config import (
    DATA_DIR, FIGURES_DIR, RESULTS_DIR,
    PROJECTION_END_YEAR, STUDY_BBOX,
)
from src.projection import (
    build_future_grid, project,
    percent_change_summary, identify_hotspots,
)
from src.visualization import plot_projection_hotspots

In [ ]:
model = joblib.load(RESULTS_DIR / 'rf_ndsi_model.joblib')
print(f'Loaded model: {type(model).__name__}')

per_point_path = DATA_DIR / 'ndsi_per_point_2000_2026.csv'
if per_point_path.exists():
    hist = pd.read_csv(per_point_path, parse_dates=['date'])
    if 'ndsi_mean' in hist.columns and 'ndsi' not in hist.columns:
        hist = hist.rename(columns={'ndsi_mean': 'ndsi'})
else:
    hist = pd.read_csv(DATA_DIR / 'ndsi_timeseries_2000_2026.csv', parse_dates=['date']) \
             .rename(columns={'ndsi_mean': 'ndsi'})
    hist['Latitude']  = (STUDY_BBOX[1] + STUDY_BBOX[3]) / 2
    hist['Longitude'] = (STUDY_BBOX[0] + STUDY_BBOX[2]) / 2
    print('⚠️  Using regional-mean fallback for the historical reference.')

print(f'Historical record: {len(hist)} rows, '
      f'{hist["date"].min().date()} → {hist["date"].max().date()}')

## 1 · Build the future projection grid

We reuse the *historical* sampling locations (so the spatial sampling design is held constant) and project monthly from 2026 to 2050.

In [ ]:
locations = (
    hist[['Latitude', 'Longitude']]
      .drop_duplicates()
      .reset_index(drop=True)
)
print(f'Unique locations in historical record: {len(locations)}')

future = build_future_grid(
    locations,
    start_year=2026,
    end_year=PROJECTION_END_YEAR,
)
print(f'Future grid points (locations × months × years): {len(future):,}')

## 2 · Predict 2026 – 2050 NDSI

In [ ]:
proj = project(model, future)
print(f'Projected {len(proj):,} (location × month × year) points')
print(f'Projected NDSI range: {proj["ndsi_pred"].min():.4f} → {proj["ndsi_pred"].max():.4f}')
print(f'Projected NDSI mean:  {proj["ndsi_pred"].mean():.4f}')

# Persist the full projection grid for any downstream district analysis
proj_csv = RESULTS_DIR / 'projection_2026_2050.csv'
proj.to_csv(proj_csv, index=False)
print(f'\nSaved: {proj_csv}')

## 3 · Domain-mean percent change

The manuscript headline: **+20.8 % domain-mean NDSI increase by 2050**.

In [ ]:
summary = percent_change_summary(hist, proj)
print(summary.to_string(index=False))

summary.to_csv(RESULTS_DIR / 'projection_percent_change_summary.csv', index=False)

## 4 · Identify chronic-salinity hotspots (top 10 %)

Locations whose mean 2026 – 2050 projected NDSI sits at or above the 90th percentile.

In [ ]:
hotspots = identify_hotspots(proj, percentile=90)
print(f'Hotspot locations: {len(hotspots)}')
print(f'Cutoff (90th-percentile NDSI): {hotspots["cutoff_value"].iloc[0]:.4f}')
hotspots.head(10)

In [ ]:
hotspots.to_csv(RESULTS_DIR / 'hotspots_top10pct_2050.csv', index=False)
print('Saved: results/hotspots_top10pct_2050.csv')

## 5 · Hotspot map (Figure of merit)

In [ ]:
fig_path = FIGURES_DIR / 'fig_2050_hotspots.png'
plot_projection_hotspots(hotspots, save_to=fig_path)
print(f'Saved: {fig_path}')

---

**Pipeline complete.** All six notebooks have been run. Final outputs:

| Artefact | Path |
|---|---|
| Validation scatter | `figures/fig_validation.png` |
| 26-yr time-series | `figures/fig_long_timeseries.png` |
| Cyclone forest plot | `figures/fig_cyclone_impacts.png` |
| Monthly climatology | `figures/fig_monthly_climatology.png` |
| August anomaly | `figures/fig_august_anomaly.png` |
| RF feature importance | `figures/fig_feature_importance.png` |
| 2050 hotspot map | `figures/fig_2050_hotspots.png` |
| RF model | `results/rf_ndsi_model.joblib` |
| Projection grid | `results/projection_2026_2050.csv` |
| Hotspot table | `results/hotspots_top10pct_2050.csv` |

## Caveats

The +20.8 % projection assumes continuation of the observed climatic regime — cyclone landfalls at historical cadence, no abrupt non-linearities. A CMIP6-conditioned (SSP2-4.5 / SSP5-8.5) extension is the natural follow-on. See `docs/methodology.md` § 8 for full assumptions.